In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/saurabhkumargupta23/shl-dataset/dataset/sample_submission.csv
/kaggle/input/datasets/saurabhkumargupta23/shl-dataset/dataset/train.csv
/kaggle/input/datasets/saurabhkumargupta23/shl-dataset/dataset/test.csv
/kaggle/input/datasets/saurabhkumargupta23/shl-dataset/dataset/audios_test/audio_885.wav
/kaggle/input/datasets/saurabhkumargupta23/shl-dataset/dataset/audios_test/audio_698.wav
/kaggle/input/datasets/saurabhkumargupta23/shl-dataset/dataset/audios_test/audio_1176.wav
/kaggle/input/datasets/saurabhkumargupta23/shl-dataset/dataset/audios_test/audio_1215.wav
/kaggle/input/datasets/saurabhkumargupta23/shl-dataset/dataset/audios_test/audio_66.wav
/kaggle/input/datasets/saurabhkumargupta23/shl-dataset/dataset/audios_test/audio_386.wav
/kaggle/input/datasets/saurabhkumargupta23/shl-dataset/dataset/audios_test/audio_1026.wav
/kaggle/input/datasets/saurabhkumargupta23/shl-dataset/dataset/audios_test/audio_330.wav
/kaggle/input/datasets/saurabhkumargupta23/shl-dataset/d

In [2]:
!pip install -q openai-whisper
!pip install -q language-tool-python
!pip install -q xgboost
!pip install -q textstat

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 10.5 MB/s eta 0:00:0000:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 9.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 kB 3.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 32.3 MB/s eta 0:00:00a 0:00:01


In [3]:
import os
import re
import numpy as np
import pandas as pd
import whisper
import language_tool_python
import textstat

from tqdm import tqdm
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

In [4]:
import pandas as pd

base_path = "/kaggle/input/datasets/saurabhkumargupta23/shl-dataset/dataset"

train_df = pd.read_csv(f"{base_path}/train.csv")
test_df = pd.read_csv(f"{base_path}/test.csv")
sample_submission = pd.read_csv(f"{base_path}/sample_submission.csv")

train_df.head()


,filename,label
0,audio_1261.wav,1.0
1,audio_942.wav,1.5
2,audio_1110.wav,1.5
3,audio_1024.wav,1.5
4,audio_538.wav,2.0


In [5]:
whisper_model = whisper.load_model("base")
tool = language_tool_python.LanguageTool('en-US')

100%|████████████████████████████████████████| 139M/139M [00:00<00:00, 295MiB/s]


In [8]:
def extract_advanced_features(audio_path):
    
    result = whisper_model.transcribe(audio_path, fp16=False)
    text = result["text"].strip()
    segments = result["segments"]
    
    words = text.split()
    sentences = re.split(r'[.!?]', text)
    
    total_words = len(words)
    total_sentences = len([s for s in sentences if s.strip()])
    
    # Grammar errors
    matches = tool.check(text)
    num_errors = len(matches)
    
    # Derived metrics
    error_density = num_errors / total_words if total_words else 0
    avg_sentence_length = total_words / total_sentences if total_sentences else 0
    lexical_diversity = len(set(words)) / total_words if total_words else 0
    
    # Fluency proxy using Whisper segments
    pauses = len(segments)
    pause_density = pauses / total_sentences if total_sentences else 0
    
    # Readability
    readability = textstat.flesch_reading_ease(text)
    
    # Repetition ratio
    repetition_ratio = 1 - lexical_diversity
    
    return {
        "total_words": total_words,
        "total_sentences": total_sentences,
        "num_errors": num_errors,
        "error_density": error_density,
        "avg_sentence_length": avg_sentence_length,
        "lexical_diversity": lexical_diversity,
        "pause_density": pause_density,
        "readability": readability,
        "repetition_ratio": repetition_ratio
    }

In [9]:
from tqdm import tqdm

train_features = []

for _, row in tqdm(train_df.iterrows(), total=len(train_df)):
    
    path = f"{base_path}/audios_train/{row['filename']}"
    
    try:
        features = extract_advanced_features(path)
        features["label"] = row["label"]
        train_features.append(features)
        
    except Exception as e:
        print(f"Error processing {row['filename']}: {e}")

100%|██████████| 444/444 [2:03:53<00:00, 16.74s/it]  


In [14]:
train_feat_df = pd.DataFrame(train_features)

X = train_feat_df.drop("label", axis=1)
y = train_feat_df["label"]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [15]:
model = XGBRegressor(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(X_train, y_train)

val_preds = model.predict(X_val)
mse = mean_squared_error(y_val, val_preds)

print("Validation MSE:", mse)

Validation MSE: 1.2006875869741753


In [17]:
from tqdm import tqdm
import os

test_features = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    
    path = f"{base_path}/audios_test/{row['filename']}"
    
    # Safety check (very important)
    if not os.path.exists(path):
        print("Missing file:", path)
        continue
    
    try:
        features = extract_advanced_features(path)
        test_features.append(features)
        
    except Exception as e:
        print(f"Error processing {row['filename']}: {e}")

100%|██████████| 195/195 [45:57<00:00, 14.14s/it] 


In [18]:
test_feat_df = pd.DataFrame(test_features)

X_test = test_feat_df

preds = model.predict(X_test)

submission = pd.DataFrame({
    "filename": test_df["filename"],
    "label": preds
})

submission.to_csv("submission.csv", index=False)

submission.head()

,filename,label
0,audio_706.wav,3.046294
1,audio_800.wav,3.638973
2,audio_68.wav,3.611071
3,audio_1267.wav,3.798203
4,audio_683.wav,2.404651
